In [ ]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import nltk
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import string
import gensim

In [ ]:
file_path = '../train.csv'
data = pd.read_csv(file_path)
df = pd.DataFrame(data)
#print(f"Data shape: {df.shape}\n")
#print(df.head())

# data checks
#print(f"Number of missing entries: {df.isna().sum()}\n") #1 for question1, 2 for question2
df.dropna(inplace = True)
#print(f"Number of missing entries: {df.isna().sum()}\n") #1 for question1, 2 for question2
#print(f"Number of duplicated entries: {df.duplicated(subset = ['qid1', 'qid2']).sum()}\n")

#getting list of unique questions and qid
def unique_qn(df): 
    """Flattens [qid1, qid2, question1, question2] into [qid, question] in ascending order."""
    df1 = df[['qid1', 'question1']].rename(columns = {'qid1': 'qid', 'question1': 'question'})
    df2 = df[['qid2', 'question2']].rename(columns = {'qid2': 'qid', 'question2': 'question'})
    combined_df = pd.concat([df1, df2], axis = 0)
    unique_questions = combined_df.drop_duplicates(subset=['qid']).sort_values('qid', ascending=True).reset_index(drop=True)
    return unique_questions

df = unique_qn(df)
print(df.head())

In [ ]:
# embedding 
questions = []
for idx, val in df.iterrows(): 
    questions.append(val['question'])

model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(questions, batch_size = 32)

In [ ]:
#Lemmatization
nltk.download("wordnet")
nltk.download("omw-1.4")

wnl = WordNetLemmatizer()

def lemmatizer(sentence):
    l_words = []
    sentence_no_punct = sentence.translate(str.maketrans("","",string.punctuation))
    word_tokens = word_tokenize(sentence_no_punct)
    for word in word_tokens:
        l_words.append(wnl.lemmatize(word,pos="v"))
    
    return l_words

#Word2Vec model
unique = unique_qn(df)
sentences = unique.iloc[0,2]
data = []

for sentence in sentences:
    data.append(lemmatizer(sentence))

model1 = gensim.models.Word2Vec(data, min_count=1,
                                vector_size=100, window=5)

